# Phase 4 D13: local models translate math and logic tasks (Kaggle, GPU T4 x2)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only**.

Runs **qwen3:1.7b**, **gemma3:1b**, **qwen2.5-coder:1.5b**, **phi3.5:3.8b**, **gemma3:4b** on the **60 dev math + logic tasks** (no test tasks, no gold answers in this notebook). Each model only *translates* (math -> one arithmetic expression, logic -> a JSON spec); the solving and checking are done offline in Python. Temperature 0; qwen3 with thinking off. Expect roughly 15-30 min in total.

At the end download **`phase4_solver_translate.zip`** from the Output panel and put it in `eval/results/phase4/`.

In [ ]:
%cd /kaggle/working
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

In [ ]:
%%writefile translate_prompts.json
{
 "math": "You translate a word problem into ONE arithmetic expression. You do not solve it.\nRules:\n- Use only numbers that appear in the problem, plus + - * / ( ) and ** for powers.\n- Write a percentage as a decimal: 25% is 0.25. \"25% off\" is (1 - 0.25). \"grows 25%\" is (1 + 0.25).\n- Use every number in the problem exactly as needed. Do not round. Do not write units.\n- Reply with exactly one line: EXPR: <expression>\n\nExample 1\nProblem: A tank holds 800 litres. 25% leaks out, then 40 litres are added. How many litres are in the tank?\nEXPR: 800 * (1 - 0.25) + 40\n\nExample 2\nProblem: Split 360 rupees in the ratio 2:7. How much is the smaller share?\nEXPR: 360 * 2 / (2 + 7)\n\nExample 3\nProblem: Mia buys 3 pens at Rs 12 each and 2 books at Rs 150 each, and pays with a Rs 500 note. How much change?\nEXPR: 500 - (3 * 12 + 2 * 150)\n\nExample 4\nProblem: A machine makes 90 parts every 15 minutes. How many parts in 3 hours?\nEXPR: 90 / 15 * 60 * 3\n\nExample 5\nProblem: A town of 5,000 people grows 10% every year. How many people live there after 2 years?\nEXPR: 5000 * (1 + 0.1) ** 2",
 "logic": "You translate a logic puzzle into JSON. You do not solve it. Reply with only the JSON object.\nSchema:\n{\"mode\": \"order\" or \"match\",\n \"entities\": [the people, exactly as named in the puzzle],\n \"items\": [the things being matched, exactly as named; only for mode \"match\"],\n \"constraints\": [one object per fact, in the order given],\n \"question\": {...}}\nMode \"order\": people are ranked 1..N (1 = first). Constraint types:\n {\"type\":\"at\",\"who\":X,\"pos\":k}            X is in position k\n {\"type\":\"not_at\",\"who\":X,\"pos\":k}        X is not in position k\n {\"type\":\"before\",\"who\":X,\"other\":Y}      X is ahead of Y (X beat Y)\n {\"type\":\"between\",\"who\":X,\"a\":Y,\"b\":Z}   X is somewhere between Y and Z\n {\"type\":\"one_between\",\"a\":X,\"b\":Y}       exactly one person is between X and Y\n {\"type\":\"right_after\",\"who\":X,\"other\":Y} X is immediately after Y\n question: {\"type\":\"who_at\",\"pos\":k}\nMode \"match\": each person has a different item. Constraint types:\n {\"type\":\"has\",\"who\":X,\"item\":i}          X has item i\n {\"type\":\"not_has\",\"who\":X,\"item\":i}      X does not have item i\n {\"type\":\"either\",\"item\":i,\"a\":X,\"b\":Y}   item i belongs to X or Y\n question: {\"type\":\"who_has\",\"item\":i}\nWrite one constraint per sentence of the puzzle (skip the introduction and the question).\n\nExample (order)\nPuzzle: Three runners - Ann, Bo and Cy - finished a race with no ties. Ann beat Bo. Cy did not finish first. Bo did not finish second. Who finished first?\n{\"mode\":\"order\",\"entities\":[\"Ann\",\"Bo\",\"Cy\"],\"constraints\":[{\"type\":\"before\",\"who\":\"Ann\",\"other\":\"Bo\"},{\"type\":\"not_at\",\"who\":\"Cy\",\"pos\":1},{\"type\":\"not_at\",\"who\":\"Bo\",\"pos\":2}],\"question\":{\"type\":\"who_at\",\"pos\":1}}\n\nExample (match)\nPuzzle: Three friends - Ann, Bo and Cy - each own a different pet: cat, dog, fish. Ann owns the cat. The fish belongs to either Bo or Cy. Cy does not own the fish. Who owns the dog?\n{\"mode\":\"match\",\"entities\":[\"Ann\",\"Bo\",\"Cy\"],\"items\":[\"cat\",\"dog\",\"fish\"],\"constraints\":[{\"type\":\"has\",\"who\":\"Ann\",\"item\":\"cat\"},{\"type\":\"either\",\"item\":\"fish\",\"a\":\"Bo\",\"b\":\"Cy\"},{\"type\":\"not_has\",\"who\":\"Cy\",\"item\":\"fish\"}],\"question\":{\"type\":\"who_has\",\"item\":\"dog\"}}"
}

In [ ]:
%%writefile tasks_translate.json
[
 {
  "id": "math-003",
  "category": "math",
  "prompt": "Quick one for the inventory report - we opened the quarter with 1940 bottles of juice. Week 1 we lost 40% of that to sales, week 2 we sold 50% of whatever was left, and in week 3 a further 45 went out. What's the closing stock?"
 },
 {
  "id": "math-004",
  "category": "math",
  "prompt": "₹8,500 is invested at 20% compound interest per year. After two years, an extra ₹1,500 is added, and the whole balance then grows for one more year at the same rate. What is the final balance?"
 },
 {
  "id": "math-005",
  "category": "math",
  "prompt": "Work out how many bags of rice remain. Starting stock: 1900. Monday sales: 25% of the stock. Tuesday sales: 10 units."
 },
 {
  "id": "math-007",
  "category": "math",
  "prompt": "A prize of ₹672 is shared among three people in the ratio 3:4:5. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-008",
  "category": "math",
  "prompt": "A prize of ₹336 is shared among three people in the ratio 1:2:4. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-010",
  "category": "math",
  "prompt": "drove 180 km doing 45 km/h, then 180 km at 30 km/h. total time on the road in hours?"
 },
 {
  "id": "math-011",
  "category": "math",
  "prompt": "A recipe that serves 4 people uses 300 g of flour. How many grams of flour are needed to serve 10 people with the same recipe?"
 },
 {
  "id": "math-014",
  "category": "math",
  "prompt": "our village has 8,800 residents and it keeps growing by 25% a year. what will the population be 2 years from now?"
 },
 {
  "id": "math-015",
  "category": "math",
  "prompt": "A shop has 1680 chairs. It sells 50% of them on Monday and 40 more on Tuesday. How many chairs are left?"
 },
 {
  "id": "math-017",
  "category": "math",
  "prompt": "Quick one for the inventory report - we opened the quarter with 900 tickets. Week 1 we lost 40% of that to sales, week 2 we sold 20% of whatever was left, and in week 3 a further 25 went out. What's the closing stock?"
 },
 {
  "id": "math-018",
  "category": "math",
  "prompt": "For the board deck: we began with 11,400 subscribers; year 1 grew 25%, year 2 shrank 20%, year 3 grew 10%. What is the year-3 subscriber count?"
 },
 {
  "id": "math-019",
  "category": "math",
  "prompt": "A prize of ₹100 is shared among three people in the ratio 1:3:6. How much more does the person with the largest share get than the person with the smallest?"
 },
 {
  "id": "math-022",
  "category": "math",
  "prompt": "bought a lamp wholesale at ₹1400, tagged it 40% higher. how many rupees of markup is that?"
 },
 {
  "id": "math-024",
  "category": "math",
  "prompt": "Split 300 rupees in the ratio 2:3. Report the bigger share."
 },
 {
  "id": "math-025",
  "category": "math",
  "prompt": "The price of a bike is raised by 20% and then the new price is cut by 20%. If the bike started at ₹500, what does it cost at the end?"
 },
 {
  "id": "math-026",
  "category": "math",
  "prompt": "Shift summary needed: P ran for the full 3 hours at 12 parts/hour; Q joined for just the final 2 hours at 10 parts/hour. Total parts from both?"
 },
 {
  "id": "math-028",
  "category": "math",
  "prompt": "i handed over ₹500 for 7 notebooks that cost ₹60 apiece - what should i get back?"
 },
 {
  "id": "math-030",
  "category": "math",
  "prompt": "A company has 7,000 subscribers. Over three consecutive years its subscriber count changes by +25%, -50% and +20% respectively. How many subscribers does it have at the end of year three?"
 },
 {
  "id": "math-031",
  "category": "math",
  "prompt": "Planning my savings: ₹9,500 sits in an account earning 20% a year, compounded annually. At the end of year 2 I top it up with ₹1,000, and leave everything in for year 3. What's the balance after year 3?"
 },
 {
  "id": "math-032",
  "category": "math",
  "prompt": "A customer purchases 6 units of item P at ₹45 and 2 units of item Q at ₹125. A 25% discount applies to the total. If the customer tenders ₹2000, compute the change returned."
 },
 {
  "id": "math-036",
  "category": "math",
  "prompt": "so Vikram's store started the week with 680 oranges, sold 25% on Monday, then another 60 on Tuesday. how many are still on the shelf?"
 },
 {
  "id": "math-037",
  "category": "math",
  "prompt": "me and my cousin are splitting ₹480 in the ratio 3 to 5 (he gets the bigger part). how much is his share?"
 },
 {
  "id": "math-038",
  "category": "math",
  "prompt": "Shift summary needed: P ran for the full 2 hours at 15 parts/hour; Q joined for just the final 1 hours at 16 parts/hour. Total parts from both?"
 },
 {
  "id": "math-041",
  "category": "math",
  "prompt": "Divya is 4 times as old as Divya's daughter. In 17 years Divya will be 3 times as old as the daughter will be then. How old is the daughter now?"
 },
 {
  "id": "math-043",
  "category": "math",
  "prompt": "i handed over ₹1000 for 3 notebooks that cost ₹18 apiece - what should i get back?"
 },
 {
  "id": "math-044",
  "category": "math",
  "prompt": "Section A has 30 students with an average mark of 59. Section B has 30 students with an average mark of 65. What is the average mark of all 60 students together?"
 },
 {
  "id": "math-045",
  "category": "math",
  "prompt": "our press stamps out 336 bolts every 12 min. running it nonstop for 4 hours, how many bolts is that?"
 },
 {
  "id": "math-046",
  "category": "math",
  "prompt": "A town has 6,800 people and its population grows 20% every year. How many people live there after 2 years?"
 },
 {
  "id": "math-049",
  "category": "math",
  "prompt": "our press stamps out 136 bolts every 8 min. running it nonstop for 2 hours, how many bolts is that?"
 },
 {
  "id": "math-050",
  "category": "math",
  "prompt": "Rohan buys 4 pens at ₹45 each and 5 folders at ₹80 each. The whole bill gets a 25% discount. Rohan pays with a ₹2000 note. How much change comes back?"
 },
 {
  "id": "logic-002",
  "category": "logic",
  "prompt": "Three runners - Gus, Kofi and Jia - finished a race with no ties. Kofi did not finish first. Kofi did not finish second. Jia did not finish first.\n\nWho finished first?"
 },
 {
  "id": "logic-003",
  "category": "logic",
  "prompt": "Four runners - Fay, Omar, Pia and Uma - finished a race with no ties. Fay did not finish fourth. Omar finished ahead of Fay. Uma finished third.\n\nWho finished fourth?"
 },
 {
  "id": "logic-004",
  "category": "logic",
  "prompt": "Five runners - Hana, Dia, Ben, Fay and Lena - finished a race with no ties.\n\n1. Hana did not finish second.\n2. Dia finished somewhere between Hana and Ben.\n3. Lena did not finish third.\n4. Lena did not finish second.\n5. Fay beat Ben.\n6. Ben did not finish fourth.\n7. Dia did not finish third.\n8. Fay did not finish second.\n\nWho finished first?"
 },
 {
  "id": "logic-006",
  "category": "logic",
  "prompt": "Five runners - Cy, Raj, Kofi, Fay and Pia - finished a race with no ties.\n\n- Kofi finished somewhere between Cy and Pia.\n- Raj beat Fay.\n- Raj did not finish third.\n- Pia finished immediately after Fay.\n- Kofi finished ahead of Pia.\n- Raj did not finish second.\n\nWho finished fourth?"
 },
 {
  "id": "logic-009",
  "category": "logic",
  "prompt": "Five runners - Kofi, Ben, Ana, Raj and Pia - finished a race with no ties.\n\n- Exactly one runner finished between Ana and Raj.\n- Kofi finished ahead of Raj.\n- Raj did not finish fourth.\n- Kofi did not finish first.\n- Raj finished ahead of Pia.\n- Ana did not finish first.\n\nWho finished second?"
 },
 {
  "id": "logic-010",
  "category": "logic",
  "prompt": "Five friends - Maya, Raj, Jia, Sara and Pia - each own a different pet: cat, dog, rabbit, fish, parrot. Sara does not own the parrot. The rabbit belongs to either Maya or Pia. The fish belongs to either Jia or Pia. The cat belongs to either Sara or Pia. Raj does not own the parrot. The fish belongs to either Jia or Sara. The rabbit belongs to either Maya or Raj.\n\nWho owns the rabbit?"
 },
 {
  "id": "logic-011",
  "category": "logic",
  "prompt": "Four runners - Raj, Fay, Lena and Cy - finished a race with no ties. Lena finished third. Cy did not finish fourth. Fay finished second.\n\nWho finished first?"
 },
 {
  "id": "logic-012",
  "category": "logic",
  "prompt": "Four friends - Ivan, Eli, Ben and Kofi - each own a different pet: dog, rabbit, cat, fish.\n\n- Kofi is not the owner of the fish.\n- The fish belongs to either Ben or Kofi.\n- Kofi does not own the cat.\n- Eli owns the rabbit.\n\nWho owns the cat?"
 },
 {
  "id": "logic-014",
  "category": "logic",
  "prompt": "Five runners - Lena, Omar, Hana, Cy and Noor - finished a race with no ties.\n\n- Cy did not finish fourth.\n- Lena finished ahead of Cy.\n- Lena beat Noor.\n- Noor did not finish fourth.\n- Hana beat Lena.\n- Omar finished somewhere between Lena and Cy.\n\nWho finished first?"
 },
 {
  "id": "logic-015",
  "category": "logic",
  "prompt": "Five people - Pia, Cy, Uma, Fay and Ana - stand in a single queue at a ticket counter.\n\n- Uma is in the queue between Pia and Ana.\n- Fay is in the queue between Pia and Ana.\n- Pia is behind Fay in the queue.\n- Ana is not second in line.\n- Fay is behind Cy in the queue.\n- Cy is not second in line.\n\nWho is third in line?"
 },
 {
  "id": "logic-016",
  "category": "logic",
  "prompt": "Five runners - Omar, Raj, Gus, Fay and Cy - finished a race with no ties.\n\n- Omar did not finish fourth.\n- Exactly one runner finished between Raj and Gus.\n- Cy did not finish first.\n- Raj finished somewhere between Gus and Cy.\n- Fay did not finish second.\n- Fay beat Omar.\n\nWho finished fifth?"
 },
 {
  "id": "logic-017",
  "category": "logic",
  "prompt": "Three runners - Ivan, Uma and Ana - finished a race with no ties. Ivan beat Ana. Ivan beat Uma. Uma did not finish second.\n\nWho finished first?"
 },
 {
  "id": "logic-019",
  "category": "logic",
  "prompt": "Five people - Kofi, Uma, Noor, Fay and Ben - stand in a single queue at a ticket counter.\n\n- Fay is behind Kofi in the queue.\n- Kofi is in the queue between Uma and Ben.\n- Fay is not fourth in line.\n- Noor is in the queue between Kofi and Fay.\n- Ben is not first in line.\n- Noor is not fourth in line.\n\nWho is third in line?"
 },
 {
  "id": "logic-021",
  "category": "logic",
  "prompt": "Five friends - Raj, Ivan, Lena, Ben and Sara - each own a different pet: fish, cat, rabbit, parrot, dog. Raj is not the owner of the dog. Raj is not the owner of the rabbit. Ivan does not own the rabbit. The fish belongs to either Ben or Sara. Sara is not the owner of the rabbit. The fish belongs to either Ivan or Ben. The cat belongs to either Lena or Sara.\n\nWho owns the parrot?"
 },
 {
  "id": "logic-024",
  "category": "logic",
  "prompt": "Three friends - Hana, Gus and Cy - each own a different pet: parrot, fish, rabbit.\n\n1. Gus is not the owner of the parrot.\n2. The rabbit belongs to either Gus or Cy.\n3. Gus does not own the rabbit.\n\nWho owns the parrot?"
 },
 {
  "id": "logic-025",
  "category": "logic",
  "prompt": "Pia, Raj, Jia, Eli and Ivan each picked up a different book from a shelf holding a diary, atlas, manual, novel or cookbook.\n\n1. Jia did not pick the atlas.\n2. Pia did not pick the diary.\n3. The manual went to one of Eli and Ivan.\n4. Raj did not pick the diary.\n5. Pia did not pick the novel.\n6. The atlas went to one of Jia and Eli.\n\nWho ended up with the novel?"
 },
 {
  "id": "logic-026",
  "category": "logic",
  "prompt": "At a cafe, four people - Fay, Omar, Noor and Eli - each ordered a different drink: coffee, juice, milk, lassi.\n\n1. Either Fay or Eli ordered the coffee.\n2. Eli ordered the lassi.\n3. Noor did not order the juice.\n\nWho ordered the coffee?"
 },
 {
  "id": "logic-029",
  "category": "logic",
  "prompt": "Three boxes are labelled \"Apples\", \"Oranges\" and \"Mixed\". Each box really contains one of: only apples, only oranges, or a mix of both, and every label is wrong. You take one fruit out of the box labelled \"Mixed\" and it is an apple. What does the box labelled \"Oranges\" contain? (answer: apples, oranges or mixed)"
 },
 {
  "id": "logic-034",
  "category": "logic",
  "prompt": "Three neighbours - Hari, Isha and Jay - each keep a different pet (cat, dog or fish) and each drink a different beverage (milk, tea or soda). Hari keeps the cat. The dog owner drinks tea. Isha drinks soda. Jay does not keep the fish. Who keeps the fish and what do they drink? Answer with the person's name only."
 },
 {
  "id": "logic-037",
  "category": "logic",
  "prompt": "Four people - Ben, Gus, Uma and Maya - stand in a single queue at a ticket counter.\n\n- Ben is behind Uma in the queue.\n- Ben and Uma have exactly one person between them.\n- Gus is in the queue between Ben and Maya.\n\nWho is fourth in line?"
 },
 {
  "id": "logic-038",
  "category": "logic",
  "prompt": "At a cafe, three people - Eli, Fay and Gus - each ordered a different drink: lassi, juice, tea. Either Fay or Gus ordered the tea. Either Eli or Fay ordered the lassi. Either Fay or Gus ordered the lassi.\n\nWho ordered the tea?"
 },
 {
  "id": "logic-040",
  "category": "logic",
  "prompt": "At a cafe, five people - Lena, Gus, Tomas, Eli and Omar - each ordered a different drink: coffee, tea, milk, lassi, juice.\n\n1. Omar did not order the lassi.\n2. Tomas did not order the coffee.\n3. Lena did not order the coffee.\n4. Either Gus or Eli ordered the milk.\n5. Lena did not order the tea.\n6. Eli did not order the coffee.\n7. Omar did not order the coffee.\n8. Omar did not order the tea.\n\nWho ordered the lassi?"
 },
 {
  "id": "logic-042",
  "category": "logic",
  "prompt": "Five friends - Lena, Ben, Noor, Ivan and Maya - each own a different pet: rabbit, fish, parrot, cat, dog.\n\n- The cat belongs to either Ben or Maya.\n- The fish belongs to either Lena or Noor.\n- The parrot belongs to either Ivan or Maya.\n- Ivan is not the owner of the rabbit.\n- Lena is not the owner of the rabbit.\n- The parrot belongs to either Ben or Maya.\n\nWho owns the fish?"
 },
 {
  "id": "logic-044",
  "category": "logic",
  "prompt": "Three runners - Lena, Fay and Cy - finished a race with no ties.\n\n1. Fay did not finish second.\n2. Lena beat Cy.\n3. Fay did not finish first.\n\nWho finished first?"
 },
 {
  "id": "logic-045",
  "category": "logic",
  "prompt": "At a cafe, three people - Sara, Cy and Ivan - each ordered a different drink: coffee, lassi, milk. Either Sara or Cy ordered the milk. Sara did not order the milk. Either Cy or Ivan ordered the lassi.\n\nWho ordered the lassi?"
 },
 {
  "id": "logic-046",
  "category": "logic",
  "prompt": "Five friends - Raj, Ben, Eli, Ana and Pia - each own a different pet: cat, fish, rabbit, dog, parrot. The cat belongs to either Raj or Ben. Raj does not own the parrot. Pia is not the owner of the fish. Ana does not own the fish. Pia does not own the rabbit. The parrot belongs to either Raj or Eli. Raj is not the owner of the cat.\n\nWho owns the cat?"
 },
 {
  "id": "logic-047",
  "category": "logic",
  "prompt": "At a cafe, four people - Jia, Uma, Ben and Gus - each ordered a different drink: tea, milk, lassi, juice.\n\n1. Either Jia or Gus ordered the lassi.\n2. Jia ordered the milk.\n3. Ben did not order the tea.\n\nWho ordered the juice?"
 },
 {
  "id": "logic-048",
  "category": "logic",
  "prompt": "Three people - Tomas, Omar and Pia - stand in a single queue at a ticket counter.\n\n- Omar is not second in line.\n- Tomas is not second in line.\n- Omar is behind Pia in the queue.\n\nWho is first in line?"
 },
 {
  "id": "logic-049",
  "category": "logic",
  "prompt": "An office has three staff - Sara, Maya and Dia - and each sits in a different coloured room (white, blue, green).\n\n- The blue room is occupied by Sara or Dia (not by anyone else).\n- The blue room is occupied by Maya or Dia (not by anyone else).\n- Maya is not in the green room.\n\nWhose is the blue room?"
 },
 {
  "id": "logic-050",
  "category": "logic",
  "prompt": "Five friends - Hana, Pia, Eli, Uma and Cy - each own a different pet: cat, parrot, rabbit, fish, dog.\n\n- Cy does not own the parrot.\n- Cy is not the owner of the dog.\n- Uma does not own the parrot.\n- The rabbit belongs to either Hana or Uma.\n- The rabbit belongs to either Hana or Pia.\n- The cat belongs to either Pia or Eli.\n- Pia is not the owner of the parrot.\n\nWho owns the dog?"
 }
]

In [ ]:
%%writefile translate_run.py
"""Phase 4 D13: local models TRANSLATE dev math/logic tasks into an expression / JSON spec. Python solves offline.
Temperature 0. Output: JSONL, one reply per line. Resumable."""
import argparse, json, os, shutil, subprocess, sys, time, urllib.request

HOST = "http://127.0.0.1:11434"
SYSTEM = json.load(open("translate_prompts.json", encoding="utf-8"))
NUM_PREDICT = {"math": 256, "logic": 800}


def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)


def install_and_serve():
    if not shutil.which("ollama"):
        sh("apt-get update -qq && apt-get install -y -qq zstd >/dev/null 2>&1 || true")
        sh("curl -fsSL https://ollama.com/install.sh | sh")
    try:
        urllib.request.urlopen(HOST + "/api/version", timeout=2)
        return
    except Exception:
        pass
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(HOST + "/api/version", timeout=2)
            return
        except Exception:
            time.sleep(1)
    sys.exit("ollama server did not start; see /tmp/ollama.log")


def chat(model, category, text):
    label = "Problem: " if category == "math" else "Puzzle: "
    payload = {"model": model, "stream": False, "keep_alive": "10m",
               "messages": [{"role": "system", "content": SYSTEM[category]},
                            {"role": "user", "content": label + text}],
               "options": {"temperature": 0, "num_predict": NUM_PREDICT[category], "num_ctx": 4096}}
    if category == "logic":
        payload["format"] = "json"
    if model.startswith("qwen3"):
        payload["think"] = False
    req = urllib.request.Request(HOST + "/api/chat", json.dumps(payload).encode(), {"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=600) as r:
        d = json.load(r)
    return d, int((time.perf_counter() - t0) * 1000)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", default="tasks_translate.json")
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--out", default="translations.jsonl")
    a = ap.parse_args()
    tasks = json.load(open(a.tasks, encoding="utf-8"))
    install_and_serve()
    done = set()
    if os.path.exists(a.out):
        for line in open(a.out, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                done.add((r["model"], r["task_id"]))
    for model in a.models:
        sh("ollama pull " + model)
        n = 0
        for t in tasks:
            if (model, t["id"]) in done:
                continue
            rec = {"task_id": t["id"], "model": model, "category": t["category"], "raw_text": "", "tokens_out": None,
                   "latency_ms": None, "done_reason": None, "error": None}
            try:
                d, ms = chat(model, t["category"], t["prompt"])
                msg = d.get("message", {})
                rec.update(raw_text=msg.get("content", ""), tokens_out=d.get("eval_count"), latency_ms=ms,
                           done_reason=d.get("done_reason"))
            except Exception as e:
                rec["error"] = str(e)[:300]
            with open(a.out, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
            if n % 20 == 0:
                print(model, n, "done", flush=True)
        sh("ollama stop " + model + " || true")
        sh("ollama rm " + model)
    print("done ->", a.out)


if __name__ == "__main__":
    main()


In [ ]:
import json, collections
t = json.load(open('tasks_translate.json'))
print(len(t), collections.Counter(x['category'] for x in t))
assert len(t) == 60
assert all(set(x) == {'id', 'category', 'prompt'} for x in t)

## Run all five models (resumable: re-run this cell if the session drops)

In [ ]:
!python translate_run.py --tasks tasks_translate.json --models qwen3:1.7b gemma3:1b qwen2.5-coder:1.5b phi3.5:3.8b gemma3:4b --out translations.jsonl

## Quick look + download

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in open('translations.jsonl') if l.strip()]
df = pd.DataFrame(rows)
print('records', len(df), 'errors', df['error'].notna().sum())
print(df.groupby(['model', 'category'])[['tokens_out', 'latency_ms']].median())
print(df.groupby(['model', 'done_reason']).size())

In [ ]:
!zip -q phase4_solver_translate.zip translations.jsonl tasks_translate.json translate_prompts.json translate_run.py && ls -lh phase4_solver_translate.zip